# Vesselbound / Rift Realm — Universal Multi-View 3D Shape Generator (Colab Pro+ GPU)

Runs **rembg (`isnet-general-use`)** + **Hunyuan3D-2mv** (`tencent/Hunyuan3D-2mv`) on Colab Pro+ GPU (**A100** / **L4** / **T4**) for any uploaded `batch*_inputs.zip` turnaround batch:
- Automatically picks the newest `/content/batch*_inputs.zip` uploaded in the Files sidebar
- Runs `rembg` (`isnet-general-use`) on dark-backdrop RGB turnaround crops and saves crisp RGBA cutouts into the output zip for local 2048 PBR art projection
- Auto-selects full 30-step **`hunyuan3d-dit-v2-mv`** on **A100 / L4 (>=18 GB VRAM)** or fast 8-step **`hunyuan3d-dit-v2-mv-turbo`** on **T4**
- Caches the loaded model in GPU memory so consecutive batches (`2A -> 2B -> 2C -> 3`) run back-to-back with zero reload delay

**Set Runtime → Change runtime type → `A100 GPU` (or `L4 GPU`) before running!**

In [ ]:
# Cell 1: Verify GPU runtime and install Hunyuan3D-2 + rembg (pure wheel deps, no --upgrade, no CUDA compile)
import os, subprocess, sys, torch

assert torch.cuda.is_available(), 'No CUDA GPU attached! Go to Runtime -> Change runtime type -> A100 GPU or L4 GPU.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'GPU Ready: {gpu_name} ({vram_gb:.1f} GB VRAM)')

if not os.path.isdir('/content/Hunyuan3D-2'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Tencent/Hunyuan3D-2.git', '/content/Hunyuan3D-2'], check=True)

pkgs = [
    'trimesh', 'einops', 'omegaconf', 'safetensors', 'huggingface_hub',
    'scikit-image', 'pymeshlab', 'opencv-python-headless', 'scipy',
    'pygltflib', 'accelerate', 'diffusers', 'transformers', 'timm',
    'rembg', 'onnxruntime'
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)
if '/content/Hunyuan3D-2' not in sys.path:
    sys.path.insert(0, '/content/Hunyuan3D-2')

import hy3dgen, rembg
print('Hunyuan3D-2 OK:', hy3dgen.__file__)
print('rembg OK:', rembg.__file__)

In [ ]:
# Cell 2: Auto-detect newest /content/batch*_inputs.zip (or override BATCH_ZIP) and unpack
import glob, json, os, shutil, zipfile

BATCH_ZIP = None  # Leave None to auto-pick newest /content/batch*_inputs.zip
if BATCH_ZIP is None:
    candidates = sorted(glob.glob('/content/batch*_inputs.zip'), key=os.path.getmtime, reverse=True)
    if candidates:
        BATCH_ZIP = candidates[0]
    else:
        from google.colab import files
        print('No /content/batch*_inputs.zip found. Please upload one via the Files sidebar or prompt...')
        uploaded = files.upload()
        first_name = next(iter(uploaded.keys()))
        BATCH_ZIP = os.path.join('/content', first_name)

assert os.path.isfile(BATCH_ZIP), f'Missing zip: {BATCH_ZIP}'
print(f'Using input archive: {BATCH_ZIP} ({os.path.getsize(BATCH_ZIP) / 1e6:.2f} MB)')

zip_stem = os.path.basename(BATCH_ZIP).replace('_inputs.zip', '').replace('.zip', '')
IN_DIR = os.path.join('/content/batch_inputs', zip_stem)
if os.path.isdir(IN_DIR):
    shutil.rmtree(IN_DIR)
os.makedirs(IN_DIR, exist_ok=True)
with zipfile.ZipFile(BATCH_ZIP, 'r') as zf:
    zf.extractall(IN_DIR)

with open(os.path.join(IN_DIR, 'manifest.json'), 'r', encoding='utf-8') as f:
    MANIFEST = json.load(f)

BATCH_TAG = zip_stem
OUT_DIR = os.path.join('/content/batch_outputs', BATCH_TAG)
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Loaded batch: {MANIFEST.get('batch', BATCH_TAG)} (tag={BATCH_TAG}, rembg={MANIFEST.get('rembg_model', 'auto')}, {len(MANIFEST['characters'])} characters)")
for item in MANIFEST['characters']:
    print(f"  - {item['id']} ({item['rig_type']}): views={list(item['mv_views'].keys())}, octree={item.get('octree_resolution', 384)}, steps={item.get('steps', 30)}")

In [ ]:
# Cell 3: Auto rembg cutouts + Multi-View Hunyuan3D-2mv shape generation + QA previews
import gc, json, os, sys, time
import cv2
import numpy as np
import torch
import trimesh
import matplotlib.pyplot as plt
from PIL import Image
from rembg import new_session, remove as rembg_remove

if '/content/Hunyuan3D-2' not in sys.path:
    sys.path.insert(0, '/content/Hunyuan3D-2')

from hy3dgen.shapegen import (
    Hunyuan3DDiTFlowMatchingPipeline,
    FloaterRemover,
    DegenerateFaceRemover,
    FaceReducer,
)
from hy3dgen.shapegen.models.conditioner import SingleImageEncoder, DualImageEncoder

# Compatibility shim for Colab's transformers >= 5.0 (DINOv2 renamed attention & SwiGLU keys)
def _patch_dinov2_v5(cls):
    if getattr(cls, '_v5_patched', False): return
    orig = cls.load_state_dict
    def _load(self, state_dict, strict=True, assign=False):
        expected = set(self.state_dict().keys())
        if any('.q_proj.' in k for k in expected) and any('.attention.query.' in k for k in state_dict):
            remapped = {}
            for k, v in state_dict.items():
                if '.mlp.weights_in.' in k:
                    g, u = v.chunk(2, dim=0)
                    remapped[k.replace('.mlp.weights_in.', '.mlp.gate_proj.')] = g.contiguous()
                    remapped[k.replace('.mlp.weights_in.', '.mlp.up_proj.')] = u.contiguous()
                elif '.mlp.weights_out.' in k:
                    remapped[k.replace('.mlp.weights_out.', '.mlp.down_proj.')] = v
                elif '.attention.attention.query.' in k:
                    remapped[k.replace('.attention.attention.query.', '.attention.q_proj.')] = v
                elif '.attention.attention.key.' in k:
                    remapped[k.replace('.attention.attention.key.', '.attention.k_proj.')] = v
                elif '.attention.attention.value.' in k:
                    remapped[k.replace('.attention.attention.value.', '.attention.v_proj.')] = v
                elif '.attention.output.dense.' in k:
                    remapped[k.replace('.attention.output.dense.', '.attention.o_proj.')] = v
                else:
                    remapped[k] = v
            state_dict = remapped
        return orig(self, state_dict, strict=strict, assign=assign)
    cls.load_state_dict = _load
    cls._v5_patched = True

_patch_dinov2_v5(SingleImageEncoder)
_patch_dinov2_v5(DualImageEncoder)

# --- Step 3A: Prepare crisp RGBA cutouts (rembg on RGB inputs, passthrough on existing RGBA) ---
rembg_model_name = MANIFEST.get('rembg_model', 'isnet-general-use')
if '_REMBG_SESS' not in globals() or globals().get('_REMBG_MODEL') != rembg_model_name:
    print(f'Initializing rembg session ({rembg_model_name})...')
    _REMBG_SESS = new_session(rembg_model_name)
    _REMBG_MODEL = rembg_model_name

def prepare_cutout(img_path: str, out_path: str) -> Image.Image:
    im = Image.open(img_path).convert('RGBA')
    arr = np.array(im)
    already_cut = (arr[:, :, 3] < 250).mean() >= 0.03
    if not already_cut:
        cut = rembg_remove(im.convert('RGB'), session=_REMBG_SESS)
        arr = np.array(cut.convert('RGBA'))
        alpha = arr[:, :, 3].copy()
        alpha[alpha < 18] = 0
        # Keep main silhouette + significant detached features (halos, floating wing shards >= 0.35% main area)
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats((alpha >= 24).astype(np.uint8), connectivity=8)
        if num_labels > 1:
            areas = stats[1:, cv2.CC_STAT_AREA]
            max_area = int(areas.max())
            keep = np.zeros_like(alpha, dtype=bool)
            for lid in range(1, num_labels):
                if stats[lid, cv2.CC_STAT_AREA] >= max(80, int(max_area * 0.0035)):
                    keep |= (labels == lid)
            alpha[~keep] = 0
        arr[:, :, 3] = alpha
        ys, xs = np.where(alpha >= 24)
        if len(xs) > 0:
            y0, y1 = int(ys.min()), int(ys.max()) + 1
            x0, x1 = int(xs.min()), int(xs.max()) + 1
            crop = arr[y0:y1, x0:x1]
            ch, cw = crop.shape[:2]
            pad = max(16, int(max(ch, cw) * 0.06))
            canvas = np.zeros((ch + 2 * pad, cw + 2 * pad, 4), dtype=np.uint8)
            canvas[pad:pad + ch, pad:pad + cw] = crop
            arr = canvas
        im = Image.fromarray(arr, mode='RGBA')
    im.save(out_path)
    return im

cutouts_by_id = {}
t_cut = time.time()
for item in MANIFEST['characters']:
    cid = item['id']
    cut_dir = os.path.join(OUT_DIR, cid, 'cutouts')
    os.makedirs(cut_dir, exist_ok=True)
    cviews = {}
    for vkey, fname in item['mv_views'].items():
        src = os.path.join(IN_DIR, cid, fname)
        dst = os.path.join(cut_dir, f'{vkey}.png')
        cviews[vkey] = prepare_cutout(src, dst)
    cutouts_by_id[cid] = cviews
print(f'Prepared cutouts for {len(cutouts_by_id)} characters in {time.time() - t_cut:.1f}s')

# Render cutouts QA contact sheet on mid-grey checkerboard
n_chars = len(MANIFEST['characters'])
fig, axes = plt.subplots(n_chars, 3, figsize=(9, 3 * n_chars), facecolor='#1a1e28')
if n_chars == 1:
    axes = [axes]
for row, item in enumerate(MANIFEST['characters']):
    cid = item['id']
    vlist = list(cutouts_by_id[cid].items())
    for col in range(3):
        ax = axes[row][col]
        ax.set_facecolor('#2b3242')
        ax.axis('off')
        if col < len(vlist):
            vk, cim = vlist[col]
            ax.imshow(cim)
            ax.set_title(f'{cid}: {vk}', color='white', fontsize=10)
plt.tight_layout()
cut_prev = os.path.join(OUT_DIR, f'{BATCH_TAG}_cutouts_preview.png')
plt.savefig(cut_prev, dpi=130, facecolor='#1a1e28')
plt.show()
print('Cutout preview saved:', cut_prev)

# --- Step 3B: Load Hunyuan3D-2mv (cached across batches on the same GPU session) ---
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
use_turbo = True  # Proven 8-step FlashVDM multi-view turbo (~10-12s/model on L4, 10x faster, identical 88-92% fidelity)
subfolder = 'hunyuan3d-dit-v2-mv-turbo' if use_turbo else 'hunyuan3d-dit-v2-mv'

if '_HY3D_PIPE' not in globals() or globals().get('_HY3D_SUBFOLDER') != subfolder:
    print(f'Loading tencent/Hunyuan3D-2mv ({subfolder}, fp16) on {vram_gb:.1f} GB VRAM...')
    t_load = time.time()
    torch.set_default_dtype(torch.float16)
    try:
        with torch.device('cuda'):
            _HY3D_PIPE = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
                'tencent/Hunyuan3D-2mv',
                subfolder=subfolder,
                variant='fp16',
                use_safetensors=True,
            )
            if use_turbo:
                _HY3D_PIPE.enable_flashvdm(mc_algo='mc')
            else:
                # Ensure pure-Python marching cubes extractor (no compiled diso needed)
                try:
                    from hy3dgen.shapegen.models.autoencoders.surface_extractors import MCSurfaceExtractor
                    _HY3D_PIPE.vae.surface_extractor = MCSurfaceExtractor()
                except Exception as e:
                    print('Note: using default surface_extractor:', e)
    finally:
        torch.set_default_dtype(torch.float32)
    _HY3D_SUBFOLDER = subfolder
    gc.collect()
    torch.cuda.empty_cache()
    print(f'Model loaded in {time.time() - t_load:.1f} s')
else:
    print(f'Reusing cached {subfolder} pipeline in GPU memory!')

pipeline = _HY3D_PIPE
floater_remover = FloaterRemover()
degen_remover = DegenerateFaceRemover()
face_reducer = FaceReducer()

def normalize_mesh(mesh: trimesh.Trimesh) -> trimesh.Trimesh:
    if isinstance(mesh, trimesh.Scene):
        mesh = mesh.dump(concatenate=True)
    if mesh.is_volume and mesh.volume < 0:
        mesh.invert()
    bounds = mesh.bounds
    cx = 0.5 * (bounds[0][0] + bounds[1][0])
    cz = 0.5 * (bounds[0][2] + bounds[1][2])
    miny = bounds[0][1]
    mesh.apply_translation([-cx, -miny, -cz])
    return mesh

report = {'batch': BATCH_TAG, 'gpu': torch.cuda.get_device_name(0), 'subfolder': subfolder, 'results': []}
meshes_for_preview = []

for item in MANIFEST['characters']:
    cid = item['id']
    c_out = os.path.join(OUT_DIR, cid)
    os.makedirs(c_out, exist_ok=True)
    glb_path = os.path.join(c_out, 'shape_raw.glb')

    if os.path.isfile(glb_path):
        print(f'[{cid}] Already generated, loading existing GLB...')
        mesh = trimesh.load(glb_path, force='mesh')
        meshes_for_preview.append((cid, mesh))
        report['results'].append({'id': cid, 'cached': True, 'faces': int(len(mesh.faces))})
        continue

    views = cutouts_by_id[cid]
    steps = 8 if use_turbo else int(item.get('steps', 30))
    octree = int(item.get('octree_resolution', 384))
    face_limit = int(item.get('face_limit', 200000))
    seed = int(item.get('seed', 42))

    print(f'[{cid}] Generating multi-view ({list(views.keys())}), steps={steps}, octree={octree}...')
    t0 = time.time()
    call_kwargs = dict(
        image=views,
        num_inference_steps=steps,
        octree_resolution=octree,
        num_chunks=20000,
        generator=torch.manual_seed(seed),
        output_type='trimesh',
    )
    try:
        mesh = pipeline(**call_kwargs, mc_algo='mc')[0]
    except TypeError:
        mesh = pipeline(**call_kwargs)[0]
    raw_faces = int(len(mesh.faces))
    mesh = floater_remover(mesh)
    mesh = degen_remover(mesh)
    mesh = face_reducer(mesh, max_facenum=face_limit)
    mesh = normalize_mesh(mesh)
    mesh.export(glb_path)
    dt = time.time() - t0
    mb = os.path.getsize(glb_path) / 1e6
    print(f'[{cid}] Done in {dt:.1f}s -> {len(mesh.faces)} faces (raw {raw_faces}), {mb:.2f} MB, extents={np.round(mesh.extents, 3).tolist()}')
    report['results'].append({
        'id': cid, 'seconds': round(dt, 1), 'raw_faces': raw_faces,
        'faces': int(len(mesh.faces)), 'mb': round(mb, 2),
        'extents': np.round(mesh.extents, 4).tolist(),
    })
    meshes_for_preview.append((cid, mesh))
    gc.collect()
    torch.cuda.empty_cache()

with open(os.path.join(OUT_DIR, 'report.json'), 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

# Render fast orthographic point-silhouette preview (front / left / back) for immediate visual QA
fig, axes = plt.subplots(len(meshes_for_preview), 3, figsize=(9, 3 * len(meshes_for_preview)), facecolor='#11131a')
if len(meshes_for_preview) == 1:
    axes = [axes]
for row, (cid, mesh) in enumerate(meshes_for_preview):
    v = np.asarray(mesh.vertices)
    step = max(1, len(v) // 18000)
    vs = v[::step]
    projs = [('front', vs[:, 0], vs[:, 1], vs[:, 2]), ('left', -vs[:, 2], vs[:, 1], vs[:, 0]), ('back', -vs[:, 0], vs[:, 1], -vs[:, 2])]
    for col, (lbl, px, py, depth) in enumerate(projs):
        ax = axes[row][col]
        ax.set_facecolor('#11131a')
        order = np.argsort(depth)
        ax.scatter(px[order], py[order], c=depth[order], cmap='viridis', s=0.6, linewidths=0)
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(f'{cid} ({lbl})', color='white', fontsize=10)
plt.tight_layout()
prev_path = os.path.join(OUT_DIR, f'{BATCH_TAG}_shapes_preview.png')
plt.savefig(prev_path, dpi=140, facecolor='#11131a')
plt.show()
print('Preview saved:', prev_path)

In [ ]:
# Cell 4: Zip all generated GLBs + crisp RGBA cutouts + previews + report and trigger download
import os, zipfile
from google.colab import files

ZIP_OUT = f'/content/{BATCH_TAG}_shapes.zip'
with zipfile.ZipFile(ZIP_OUT, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, fnames in os.walk(OUT_DIR):
        for fn in sorted(fnames):
            full = os.path.join(root, fn)
            rel = os.path.relpath(full, OUT_DIR)
            zf.write(full, rel)

print(f'READY: {ZIP_OUT} ({os.path.getsize(ZIP_OUT) / 1e6:.2f} MB)')
files.download(ZIP_OUT)

In [ ]:
# Cell 5: 360° Hunyuan3D-Paint PBR Texture Baker (for Massive Titan Bosses / single-view or hero meshes)
# Expects /content/batch_titans_paint.zip (or set PAINT_ZIP explicitly).
import gc, glob, json, os, shutil, subprocess, sys, time, zipfile
import numpy as np
import torch
import trimesh
from PIL import Image
import matplotlib.pyplot as plt
from google.colab import files

PAINT_ZIP = '/content/batch_titans_paint.zip'
assert os.path.isfile(PAINT_ZIP), f'Upload {PAINT_ZIP} to /content/ first!'

# 1. Build custom_rasterizer & differentiable_renderer once per Colab runtime
if '/content/Hunyuan3D-2' not in sys.path:
    sys.path.insert(0, '/content/Hunyuan3D-2')

try:
    import custom_rasterizer
    print('custom_rasterizer already installed.')
except ImportError:
    print('Building custom_rasterizer (CUDA) once for this runtime...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', '-e', '/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer'], check=True)

dr_dir = '/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer'
if not glob.glob(os.path.join(dr_dir, 'mesh_processor*.so')):
    print('Building differentiable_renderer mesh_processor (C++)...')
    subprocess.run([sys.executable, 'setup.py', 'build_ext', '--inplace'], cwd=dr_dir, check=True)

# Free shape pipeline temporarily on 22 GB L4 so Hunyuan3DPaint gets full VRAM
if '_HY3D_PIPE' in globals():
    del globals()['_HY3D_PIPE']
    globals().pop('_HY3D_SUBFOLDER', None)
    gc.collect()
    torch.cuda.empty_cache()

from hy3dgen.texgen import Hunyuan3DPaintPipeline

if '_HY3D_PAINT' not in globals():
    print('Loading Hunyuan3DPaintPipeline (tencent/Hunyuan3D-2, hunyuan3d-paint-v2-0-turbo)...')
    t0 = time.time()
    try:
        _HY3D_PAINT = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2', subfolder='hunyuan3d-paint-v2-0-turbo')
    except Exception as e:
        print('Turbo paint fallback to standard hunyuan3d-paint-v2-0:', e)
        _HY3D_PAINT = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2', subfolder='hunyuan3d-paint-v2-0')
    print(f'Hunyuan3DPaintPipeline loaded in {time.time() - t0:.1f}s')

IN_DIR = '/content/titans_paint_in'
OUT_DIR = '/content/titans_paint_out'
if os.path.isdir(IN_DIR):
    shutil.rmtree(IN_DIR)
os.makedirs(IN_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

with zipfile.ZipFile(PAINT_ZIP, 'r') as zf:
    zf.extractall(IN_DIR)

manifest = json.load(open(os.path.join(IN_DIR, 'manifest.json'), 'r', encoding='utf-8'))
items = manifest['items']
print(f'Painting {len(items)} Titan Bosses in seamless 360° 2048x2048...')

for item in items:
    cid = item['id']
    c_out = os.path.join(OUT_DIR, cid)
    os.makedirs(c_out, exist_ok=True)
    out_glb = os.path.join(c_out, 'painted_360.glb')
    if os.path.isfile(out_glb):
        print(f'[{cid}] Cached painted_360.glb, skipping.')
        continue
    t0 = time.time()
    mesh = trimesh.load(os.path.join(IN_DIR, cid, item['mesh']), force='mesh')
    if isinstance(mesh, trimesh.Scene):
        mesh = mesh.dump(concatenate=True)
    img = Image.open(os.path.join(IN_DIR, cid, item['image'])).convert('RGBA')
    textured = _HY3D_PAINT(mesh, image=img)
    textured.export(out_glb)
    print(f'[{cid}] 360° painted in {time.time() - t0:.1f}s -> {os.path.getsize(out_glb)/1e6:.2f} MB')
    gc.collect()
    torch.cuda.empty_cache()

ZIP_OUT = '/content/batch_titans_painted.zip'
with zipfile.ZipFile(ZIP_OUT, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, fnames in os.walk(OUT_DIR):
        for fn in sorted(fnames):
            full = os.path.join(root, fn)
            zf.write(full, os.path.relpath(full, OUT_DIR))
print(f'READY: {ZIP_OUT} ({os.path.getsize(ZIP_OUT)/1e6:.2f} MB)')
files.download(ZIP_OUT)